In [1]:
# ==========================================
# CELL 1: SETUP
# ==========================================
import os
import sys
import yaml
from pathlib import Path
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

# Đảm bảo Python nhận diện được thư mục src/ trong project của bạn
sys.path.insert(0, os.path.abspath('.'))

print("✅ Đã nạp xong các thư viện cơ bản!")

✅ Đã nạp xong các thư viện cơ bản!


c:\Users\cubi8\hybrid-rag-for-BEIR\venv\Lib\site-packages\beir\util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
# ==========================================
# CELL 2: TẢI DATA VÀ CẤU HÌNH (SCIFACT)
# ==========================================
dataset = "scifact"
# Tải và giải nén thẳng vào thư mục ./data/ của project
#url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
#data_path = util.download_and_unzip(url, "data") 
# Nếu đã tải về rồi, bạn có thể chỉ định đường dẫn trực tiếp đến thư mục giải nén
data_path = f"./data"
print(f"Đang nạp dataset {dataset.upper()}...")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")
print(f"✅ Đã nạp {len(queries)} câu hỏi và {len(corpus)} tài liệu.")

# Đọc file config.yaml từ máy tính
config_path = Path("config/config.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Trỏ chính xác về Ollama Desktop đang chạy trên máy bạn
config['ollama']['base_url'] = "http://localhost:11434"

Đang nạp dataset SCIFACT...


100%|██████████| 5183/5183 [00:00<00:00, 82776.43it/s]

✅ Đã nạp 300 câu hỏi và 5183 tài liệu.


In [3]:
# ========================================================
# CELL 3: KHỞI TẠO MÔ HÌNH HYBRID (ADVANCED RAG)
# Tích hợp Document Reconstruction & Alpha Fusion
# ========================================================
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings 
from src.hybrid_rag.document_loader import DocumentLoaderUtility

# BẮT BUỘC: Gọi hàm khởi tạo đã được nâng cấp "Chén Thánh"
from src.hybrid_rag.hybrid_retriever import create_beir_hybrid_retriever 

class MyAdvancedRetriever:
    def __init__(self, data_path, config):
        print("🔧 Đang khởi tạo mô hình Advanced Hybrid RAG...")
        
        # 1. Nạp tài liệu (Sẽ giữ nguyên cấu trúc chunk ban đầu cho Vector Store)
        loader = DocumentLoaderUtility(data_path, config=config)
        documents = loader.load_documents()
        
        # 2. Khởi tạo Vector Embeddings (Đang dùng Ollama Desktop theo code cũ của bạn)
        print(f"Đang kết nối model {config['ollama']['embedding_model']} trên Localhost...")
        embeddings = OllamaEmbeddings(
            model=config['ollama']['embedding_model'],
            base_url=config['ollama']['base_url']
        )
        
        # 3. Nạp Vector Database
        persist_dir = config['vector_store']['persist_directory']
        vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)
        
        # 4. GỌI KIẾN TRÚC MỚI (Tự động chạy Document Reconstruction & IRSystem bên trong)
        self.retriever = create_beir_hybrid_retriever(
            documents=documents, 
            vectorstore=vectorstore, 
            config=config
        )
        
        # In xác nhận trọng số đang chạy
        print(f"\n✅ TRỌNG SỐ HIỆN TẠI:")
        print(f" -> Dense Weight (Vector): {self.retriever.dense_weight}")
        print(f" -> Sparse Weight (BM25+): {1.0 - self.retriever.dense_weight}")

    def search(self, query_text):
        """Hàm chuẩn hóa đầu ra cho BEIR"""
        # Với kiến trúc mới, hàm invoke đã tự động Max Pooling và trả về doc_id gốc
        docs = self.retriever.invoke(query_text)
        results = {}
        for doc in docs:
            doc_id = doc.metadata.get('doc_id')
            score = float(doc.metadata.get('hybrid_score', 0.0))
            if doc_id:
                results[doc_id] = score
        return results

# TẠO ĐỐI TƯỢNG (Sẽ mất chút thời gian để gom chunk và Index lên RAM)
my_model = MyAdvancedRetriever(data_path, config)
print("\n✅ Hệ thống đã sẵn sàng để chấm điểm!")

C:\Users\cubi8\AppData\Local\Temp\ipykernel_13188\2474583615.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


🔧 Đang khởi tạo mô hình Advanced Hybrid RAG...
✅ Loaded: corpus.jsonl (1/2)
✅ Loaded: queries.jsonl (2/2)

📚 Total files loaded: 2
📄 Total chunks created: 21873
Đang kết nối model nomic-embed-text trên Localhost...


C:\Users\cubi8\AppData\Local\Temp\ipykernel_13188\2474583615.py:29: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=persist_dir, embedding_function=embeddings)


🛠️ Đang kích hoạt tiến trình Tái cấu trúc tài liệu gốc (Document Reconstruction)...
 -> Đã gom 21873 chunks về 6292 tài liệu nguyên gốc.

✅ TRỌNG SỐ HIỆN TẠI:
 -> Dense Weight (Vector): 0.85
 -> Sparse Weight (BM25+): 0.15000000000000002

✅ Hệ thống đã sẵn sàng để chấm điểm!


In [4]:
# ==========================================
# CELL 4: VÒNG LẶP BENCHMARK SCIFACT
# ==========================================
def evaluate_model(queries_dict, model):
    results = {}
    total = len(queries_dict)
    print(f"🚀 Đang chạy truy vấn trên {total} câu hỏi...")
    
    for idx, (qid, query_text) in enumerate(queries_dict.items(), 1):
        # In tiến độ để bạn theo dõi
        if idx % 50 == 0:
            print(f" -> Đã xử lý {idx}/{total} câu hỏi...")
            
        try:
            results[qid] = model.search(query_text)
        except Exception as e:
            print(f"\n❌ LỖI NGHIÊM TRỌNG ở câu {idx} (QID: {qid}): {str(e)}")
            results[qid] = {}
            
    return results

# 1. Chạy quá trình truy xuất
model_results = evaluate_model(queries, my_model)

# 2. Chấm điểm bằng BEIR
evaluator = EvaluateRetrieval()

print("\n" + "🏆 "*15)
print("  BẢNG ĐIỂM ĐÁNH GIÁ SCIFACT (METRICS)")
print("🏆 "*15)

ndcg, _map, recall, precision = evaluator.evaluate(qrels, model_results, evaluator.k_values)
mrr = evaluator.evaluate_custom(qrels, model_results, evaluator.k_values, metric="mrr")

# In ra các chỉ số quan trọng nhất
print(f"📊 NDCG@10:   {ndcg.get('NDCG@10', 0):.5f}")
print(f"📊 MAP@10:    {_map.get('MAP@10', 0):.5f}")
print(f"📊 Recall@10: {recall.get('Recall@10', 0):.5f}")
print(f"📊 P@10:      {precision.get('P@10', 0):.5f}")
print(f"📊 MRR@10:    {mrr.get('MRR@10', 0):.5f}")
print("="*40)

🚀 Đang chạy truy vấn trên 300 câu hỏi...
 -> Đã xử lý 50/300 câu hỏi...
 -> Đã xử lý 100/300 câu hỏi...
 -> Đã xử lý 150/300 câu hỏi...
 -> Đã xử lý 200/300 câu hỏi...
 -> Đã xử lý 250/300 câu hỏi...
 -> Đã xử lý 300/300 câu hỏi...

🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 
  BẢNG ĐIỂM ĐÁNH GIÁ SCIFACT (METRICS)
🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 
📊 NDCG@10:   0.62346
📊 MAP@10:    0.55944
📊 Recall@10: 0.81300
📊 P@10:      0.08933
📊 MRR@10:    0.56603


In [5]:
# ========================================================
# ĐỊNH NGHĨA MÔ HÌNH: PHƯƠNG PHÁP TRUYỀN THỐNG (IRSystem)
# Thuật toán: BM25 (IDF Squared) + Rocchio PRF
# ========================================================
import re
import math
import tempfile
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Tải tài nguyên NLTK (chạy 1 lần)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [6]:
class IRSystem:
    def __init__(self, k1=1.0, b=0.45, fb_docs=3, fb_terms=5, alpha=0.5):
        self.k1 = k1
        self.b = b
        self.fb_docs = fb_docs
        self.fb_terms = fb_terms
        self.alpha = alpha

        self.Index = {}
        self.idf = {}
        self.doc_lengths = {}
        self.avgdl = 0
        self.N = 0
        self.doc_tokens_map = {}

        self.ps = PorterStemmer()
        self.stoplist = set(stopwords.words("english"))
        self.puncts = set(['.', ',', ':', '`', '"', "'", '!', '?', "``", "''", '(', ')', '--', ';'])

    def preprocess(self, text):
        if not text: return []
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower())
        tokens = word_tokenize(text)
        processed = []
        for tok in tokens:
            if tok not in self.stoplist and tok not in self.puncts and len(tok) > 2:
                processed.append(self.ps.stem(tok))
        return processed

    def index(self, docfile):
        total_length = 0
        self.N = 0

        with open(docfile, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split("\t")
                if len(parts) < 2: continue

                self.N += 1
                docid = parts[0]
                content = parts[1]

                terms = self.preprocess(content)
                self.doc_tokens_map[docid] = terms

                doc_len = len(terms)
                self.doc_lengths[docid] = doc_len
                total_length += doc_len

                counts = {}
                for t in terms:
                    counts[t] = counts.get(t, 0) + 1

                for t, tf in counts.items():
                    if t not in self.Index:
                        self.Index[t] = {}
                    self.Index[t][docid] = tf

        if self.N > 0:
            self.avgdl = total_length / self.N

        for term, postings in self.Index.items():
            df = len(postings)
            self.idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def compute_bm25(self, q_tf_dict):
        scores = {}
        safe_avgdl = self.avgdl if self.avgdl > 0 else 1.0

        for t, q_weight in q_tf_dict.items():
            if t in self.Index:
                idf_val = self.idf[t]
                for docid, doc_tf in self.Index[t].items():
                    dl = self.doc_lengths[docid]
                    denom = self.k1 * (1 - self.b + self.b * (dl / safe_avgdl)) + doc_tf
                    score = (idf_val ** 2) * (doc_tf * (self.k1 + 1)) / denom * q_weight
                    scores[docid] = scores.get(docid, 0) + score
        return scores

In [7]:
# --- LỚP BỌC (WRAPPER) ĐỂ KẾT NỐI IRSYSTEM VỚI BEIR ---
class MyRetriever:
    def __init__(self, corpus_dict):
        print("🔧 Đang khởi tạo mô hình Truyền thống (IRSystem)...")
        # Khởi tạo thuật toán với các tham số siêu việt của bạn
        self.ir = IRSystem(k1=1.0, b=0.45, fb_docs=3, fb_terms=5, alpha=0.5)

        # 1. Chuyển đổi Corpus của BEIR thành File tạm thời để IRSystem đọc
        print("Đang Index tài liệu trên RAM (Quá trình này tùy thuộc vào CPU)...")
        with tempfile.NamedTemporaryFile(mode='w', delete=False, encoding='utf-8') as f:
            self.temp_docfile = f.name
            for doc_id, doc_data in corpus_dict.items():
                text = f"{doc_data.get('title', '')} {doc_data.get('text', '')}"
                # Xóa \t và \n để không làm hỏng format split("\t") của hàm index
                text = text.replace('\t', ' ').replace('\n', ' ')
                f.write(f"{doc_id}\t{text}\n")

        # 2. Chạy hàm index của class
        self.ir.index(self.temp_docfile)
        print("✅ Index hoàn tất!")

    def search(self, query_text):
        """Mô phỏng lại hàm query() nhưng chạy In-memory cho 1 câu hỏi"""
        # --- Tiền xử lý ---
        q_terms = self.ir.preprocess(query_text)
        q_original_tf = {}
        for t in q_terms:
            q_original_tf[t] = q_original_tf.get(t, 0) + 1

        # --- Vòng 1: BM25 ---
        pass_1_scores = self.ir.compute_bm25(q_original_tf)
        top_docs = sorted(pass_1_scores.items(), key=lambda x: x[1], reverse=True)[:self.ir.fb_docs]

        # --- Vòng 2: Mở rộng truy vấn (PRF) ---
        expansion_candidates = {}
        for docid, _ in top_docs:
            if docid not in self.ir.doc_tokens_map: continue
            doc_terms = self.ir.doc_tokens_map[docid]
            for t in doc_terms:
                if t not in q_original_tf and t in self.ir.idf:
                    tf = self.ir.Index[t].get(docid, 0)
                    tf_weight = 1 + math.log10(tf) if tf > 0 else 0
                    expansion_candidates[t] = expansion_candidates.get(t, 0) + (tf_weight * self.ir.idf[t])

        top_expansion_terms = sorted(expansion_candidates.items(), key=lambda x: x[1], reverse=True)[:self.ir.fb_terms]

        q_expanded_tf = q_original_tf.copy()
        for term, _ in top_expansion_terms:
            q_expanded_tf[term] = self.ir.alpha

        final_scores = self.ir.compute_bm25(q_expanded_tf)

        # --- Format kết quả ---
        # Lấy top 100 thay vì 1000 để chạy nhanh hơn nhưng vẫn đủ để BEIR tính MAP chính xác
        sorted_res = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:100]
        return {k: v for k, v in sorted_res}

# KHỞI TẠO ĐỐI TƯỢNG (Colab sẽ lưu biến này để Cell 5 chấm điểm)
my_model = MyRetriever(corpus)

🔧 Đang khởi tạo mô hình Truyền thống (IRSystem)...
Đang Index tài liệu trên RAM (Quá trình này tùy thuộc vào CPU)...
✅ Index hoàn tất!


In [8]:
# ==========================================
# VÒNG LẶP BENCHMARK & ĐÁNH GIÁ METRICS
# ==========================================
def evaluate_model(queries_dict, model):
    results = {}
    total = len(queries_dict)
    print(f"🚀 Đang chạy truy vấn trên {total} câu hỏi...")

    for idx, (qid, query_text) in enumerate(queries_dict.items(), 1):
        if idx % 50 == 0:
            print(f" -> Đã xử lý {idx}/{total} câu hỏi...")

        # Gọi hàm search của mô hình
        results[qid] = model.search(query_text)

    return results

# 1. Chạy quá trình truy xuất
model_results = evaluate_model(queries, my_model)

# 2. Chấm điểm bằng BEIR
evaluator = EvaluateRetrieval()

print("\n" + "🏆 "*15)
print("  BẢNG ĐIỂM ĐÁNH GIÁ (METRICS)")
print("🏆 "*15)

ndcg, _map, recall, precision = evaluator.evaluate(qrels, model_results, evaluator.k_values)
mrr = evaluator.evaluate_custom(qrels, model_results, evaluator.k_values, metric="mrr")

# In các chỉ số Top 10
print(f"📊 NDCG@10:   {ndcg.get('NDCG@10', 0):.5f}")
print(f"📊 MAP@10:    {_map.get('MAP@10', 0):.5f}")
print(f"📊 Recall@10: {recall.get('Recall@10', 0):.5f}")
print(f"📊 P@10:      {precision.get('P@10', 0):.5f}")
print(f"📊 MRR@10:    {mrr.get('MRR@10', 0):.5f}")
print("="*40)

🚀 Đang chạy truy vấn trên 300 câu hỏi...
 -> Đã xử lý 50/300 câu hỏi...
 -> Đã xử lý 100/300 câu hỏi...
 -> Đã xử lý 150/300 câu hỏi...
 -> Đã xử lý 200/300 câu hỏi...
 -> Đã xử lý 250/300 câu hỏi...
 -> Đã xử lý 300/300 câu hỏi...

🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 
  BẢNG ĐIỂM ĐÁNH GIÁ (METRICS)
🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 🏆 
📊 NDCG@10:   0.59414
📊 MAP@10:    0.51667
📊 Recall@10: 0.81606
📊 P@10:      0.08933
📊 MRR@10:    0.52719
